# Direct execution of author GRU cells

Pinned commit: 6e8252cb9080378905b2aae30033c5946b72335b. Source cell IDs are zero-based. HPO10 then fixed recipe, original RNG order.

In [1]:
from pathlib import Path
import json
DATA_PATH = Path('D:\\UIT\\DS317\\BAF_project\\data\\raw\\Base.csv')
EXPECTED_ROWS = 1000000

## Author cell 0

In [2]:
import pandas as pd
# from google.colab import drive

# # Mount Google Drive
# drive.mount('/content/drive')

# Load dataset
# df = pd.read_csv('drive/My Drive/Colab Notebooks/Base.csv')
df = pd.read_csv(DATA_PATH)

# Specify the label column (the label column name in Base.csv is fraud_bool)
label_col = "fraud_bool"
if label_col not in df.columns:
    raise KeyError(f"Label column '{label_col}' not found. Please verify the label column name in the CSV and modify label_col accordingly.")

# Construct X, y (these variables will be used by the model later)
y = df[label_col].astype(int)
X = df.drop(columns=[label_col])

print("\nLabel distribution:")
print(y.value_counts(dropna=False))
print("Positive rate = {:.6f}".format(y.mean()))
print("\nFirst 5 rows:")
display(df.head())
# For convenience in subsequent steps, save basic information
print("\nNote: We will use variables X, y, df as the main data objects in the following steps.")



Label distribution:
fraud_bool
0    988971
1     11029
Name: count, dtype: int64
Positive rate = 0.011029

First 5 rows:

Note: We will use variables X, y, df as the main data objects in the following steps.


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,13096.035018,7850.955007,6742.080561,5,5,CB,163,1,BC,0,1,9,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,9223.283431,5745.251481,5941.664859,3,18,CA,154,1,BC,1,1,2,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,4471.472149,5471.988958,5992.555113,15,11,CA,89,1,BC,0,1,30,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,14431.993621,6755.344479,5970.336831,11,13,CA,90,1,BC,0,1,1,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,7601.511579,5124.046930,5940.734212,1,6,CA,91,0,BC,1,1,26,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


In [3]:
if len(df) != EXPECTED_ROWS or set(df.fraud_bool.unique()) != {0, 1}:
    raise ValueError('Wrong input row count or labels')


## Author cell 2

In [4]:
# Columns to exclude from features
exclude_cols = [ "prev_address_months_count", # Features with excessive missing samples that are difficult to impute
                 "month"]  # Timestamp

# === Organize num_cols/cat_cols, build ColumnTransformer, and perform stratified train/test split ===

num_cols = ['income', 'name_email_similarity', 'current_address_months_count', 'customer_age',
            'days_since_request', 'intended_balcon_amount', 'zip_count_4w', 'velocity_6h',
            'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w',
            'credit_risk_score', 'bank_months_count', 'proposed_credit_limit', 'session_length_in_minutes']

cat_cols = ['payment_type', 'employment_status', 'email_is_free', 'housing_status', 'phone_home_valid',
            'phone_mobile_valid', 'has_other_cards', 'foreign_request', 'source', 'device_os',
            'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count']

print("Final num_cols count:", len(num_cols))
print("Final cat_cols count:", len(cat_cols))

Final num_cols count: 16
Final cat_cols count: 13


## Author cell 3

In [5]:
# Train/test split (stratified split with stratify=y to prevent positive class distribution imbalance)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import sklearn

X = df.drop(columns=[label_col])
y = df[label_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Dataset split completed. Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train positives:", int(y_train.sum()), " Test positives:", int(y_test.sum()))

# Numeric features: median imputation + standardization
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical features: constant imputation + OneHot (automatically select parameters based on sklearn version)
if sklearn.__version__ >= "1.2":
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ])
else:  # Compatible with older versions
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=True))
    ])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
], remainder="drop", sparse_threshold=0.3)

# Fit preprocessor (only fit on training set to avoid data leakage), and transform train/test sets for inspection
preprocessor.fit(X_train)

X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)
print("Transformed types:", type(X_train_t), type(X_test_t))
print("Transformed shapes (train/test):", X_train_t.shape, X_test_t.shape)

# Simply extract feature names after preprocessing (for plotting feature importance later)
def get_feature_names_from_preprocessor(preproc):
    feature_names = []
    for name, trans, cols in preproc.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat":
            ohe = trans.named_steps["onehot"]
            try:
                ohe_names = ohe.get_feature_names_out(cols)
            except Exception:
                # Compatible with older sklearn versions
                ohe_names = []
                for i, col in enumerate(cols):
                    cats = ohe.categories_[i]
                    ohe_names.extend([f"{col}__{str(val)}" for val in cats])
            feature_names.extend(list(ohe_names))
    return feature_names

feat_names = get_feature_names_from_preprocessor(preprocessor)
print("Number of post-transform features (approx):", len(feat_names))
print("First 30 feature names:", feat_names[:30])

Dataset split completed. Train shape: (800000, 31) Test shape: (200000, 31)
Train positives: 8823  Test positives: 2206
Transformed types: <class 'numpy.ndarray'> <class 'numpy.ndarray'>
Transformed shapes (train/test): (800000, 59) (200000, 59)
Number of post-transform features (approx): 59
First 30 feature names: ['income', 'name_email_similarity', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'credit_risk_score', 'bank_months_count', 'proposed_credit_limit', 'session_length_in_minutes', 'payment_type_AA', 'payment_type_AB', 'payment_type_AC', 'payment_type_AD', 'payment_type_AE', 'employment_status_CA', 'employment_status_CB', 'employment_status_CC', 'employment_status_CD', 'employment_status_CE', 'employment_status_CF', 'employment_status_CG', 'email_is_free_0', 'email_is_free_1']


## Author cell 27

In [6]:
# === (1) Use fitted parts of preprocessor to build DataLoaders for GRU (direct tabular input) ===
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import joblib

# Hyperparameters (adjust as needed)
BATCH_SIZE_GRU = 256
random_state = 42

# Confirm preprocessor has been fitted
if not hasattr(preprocessor, "transformers_"):
    raise RuntimeError("preprocessor not fitted — please run preprocessor.fit(X_train) first (already done in LGBM code section)")

# 1) Get numeric pipeline (imputer+scaler) and reuse its transform
#    If you want to directly reuse preprocessor.transform(X)[...], but we only need the numeric part's transformation
num_pipeline = preprocessor.named_transformers_.get('num', None)
if num_pipeline is None:
    raise RuntimeError("No 'num' transformer in preprocessor, please check if num_cols is correct")

# 2) Get categories_ from categorical onehot encoder to generate label->index mapping
cat_transformer = preprocessor.named_transformers_.get('cat', None)
if cat_transformer is None:
    raise RuntimeError("No 'cat' transformer in preprocessor, please check if cat_cols is correct")

# onehot step:
if hasattr(cat_transformer, "named_steps") and 'onehot' in cat_transformer.named_steps:
    ohe = cat_transformer.named_steps['onehot']
else:
    raise RuntimeError("onehot step not included in categorical transformer or differently named, please check preprocessor construction")

# categories_ is list of arrays, corresponding to the order of cat_cols
ohe_categories = list(ohe.categories_)  # Each element is all categories for that column (from training set)
# Build cat -> index map for each categorical column, keep unseen mapping to len(categories)
cat_to_index = {}
cat_num_embeddings = {}
for i,c in enumerate(cat_cols):
    cats = list(ohe_categories[i])
    mapping = {str(v): idx for idx, v in enumerate(cats)}  # cat value -> 0..K-1
    cat_to_index[c] = mapping
    cat_num_embeddings[c] = len(cats) + 1  # +1 reserved for unseen

# 3) Dataset class returning input_ids=(num_tensor,cat_tensor), label
class TabularGRUDataset(Dataset):
    def __init__(self, X_df, y_ser, num_cols, cat_cols, num_pipeline, cat_to_index):
        self.X = X_df.reset_index(drop=True).copy()
        self.y = y_ser.reset_index(drop=True).astype(int).copy()
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.num_pipeline = num_pipeline
        self.cat_to_index = cat_to_index

        # numeric transformation using fitted numeric pipeline (imputer+scaler)
        if len(self.num_cols) > 0:
            # num_pipeline.transform accepts DataFrame or array
            self.num_mat = self.num_pipeline.transform(self.X[self.num_cols])
            # ensure float32
            self.num_mat = np.array(self.num_mat, dtype=np.float32)
        else:
            self.num_mat = np.zeros((len(self.X), 0), dtype=np.float32)

        # categorical -> integer indices using cat_to_index mapping
        cat_mats = []
        for c in self.cat_cols:
            vals = self.X[c].fillna("__MISSING__").astype(str).values
            map_c = self.cat_to_index[c]
            int_vals = []
            unseen_idx = len(map_c)   # reserved index for unseen
            for v in vals:
                int_vals.append(map_c.get(v, unseen_idx))
            cat_mats.append(np.array(int_vals, dtype=np.int64))
        if len(cat_mats) > 0:
            self.cat_mat = np.stack(cat_mats, axis=1)
        else:
            self.cat_mat = np.zeros((len(self.X), 0), dtype=np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        num_row = torch.from_numpy(self.num_mat[idx])    # (n_num,)
        cat_row = torch.from_numpy(self.cat_mat[idx])    # (n_cat,)
        label = torch.tensor(int(self.y.iloc[idx]), dtype=torch.long)
        return {'input_ids': (num_row, cat_row), 'label': label}

# 4) Create datasets/loaders: if X_val not present, split X_train into train+val (consistent with LGBM)
# Make sure X_train, X_test, y_train, y_test exist in workspace
if 'X_val' not in globals():
    # merge labels into X_train copy for splitting convenience
    X_train_for_split = X_train.copy()
    X_train_for_split[label_col] = y_train.values
    X_train_df, X_val_df = train_test_split(X_train_for_split, test_size=0.2, stratify=X_train_for_split[label_col], random_state=random_state)
    # separate y
    y_train_df = X_train_df[label_col]
    X_train_df = X_train_df.drop(columns=[label_col])
    y_val_df = X_val_df[label_col]
    X_val_df = X_val_df.drop(columns=[label_col])
else:
    X_train_df = X_train.copy()
    y_train_df = y_train.copy()
    X_val_df = X_val.copy()
    y_val_df = y_val.copy()

# ensure X_test has label
X_test_df = X_test.copy()
X_test_df[label_col] = y_test.values

# Build dataset objects
train_dataset_3 = TabularGRUDataset(X_train_df, y_train_df, num_cols, cat_cols, num_pipeline, cat_to_index)
val_dataset_3   = TabularGRUDataset(X_val_df, y_val_df, num_cols, cat_cols, num_pipeline, cat_to_index)
test_dataset_3  = TabularGRUDataset(X_test_df.drop(columns=[label_col]), X_test_df[label_col], num_cols, cat_cols, num_pipeline, cat_to_index)

# DataLoaders
train_loader_3 = DataLoader(train_dataset_3, batch_size=BATCH_SIZE_GRU, shuffle=True)
val_loader_3   = DataLoader(val_dataset_3, batch_size=BATCH_SIZE_GRU, shuffle=False)
test_loader_3  = DataLoader(test_dataset_3, batch_size=BATCH_SIZE_GRU, shuffle=False)

# expose cat_num_embeddings dict for model construction
# cat_num_embeddings already prepared above
print("Built GRU loaders. Sizes:", len(train_dataset_3), len(val_dataset_3), len(test_dataset_3))
print("Categorical embedding sizes (per column):")
print({k: cat_num_embeddings[k] for k in list(cat_num_embeddings)[:10]})


Built GRU loaders. Sizes: 640000 160000 200000
Categorical embedding sizes (per column):
{'payment_type': 6, 'employment_status': 8, 'email_is_free': 3, 'housing_status': 8, 'phone_home_valid': 3, 'phone_mobile_valid': 3, 'has_other_cards': 3, 'foreign_request': 3, 'source': 3, 'device_os': 6}


## Author cell 28

In [7]:


# Define GRU Model
import torch.nn as nn
class GRUSentimentModel(nn.Module):
    def __init__(self, num_cols, cat_cols, cat_num_embeddings, embedding_dim=128, hidden_dim=256, output_dim=2, dropout=0.3):
        super(GRUSentimentModel, self).__init__()
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.n_num = len(num_cols)
        self.n_cat = len(cat_cols)
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        # numeric projection: shared linear (scalar -> embedding_dim)
        self.num_proj = nn.Linear(1, embedding_dim)
        # categorical embeddings: one embedding per categorical feature
        self.cat_embeddings = nn.ModuleDict({
            c: nn.Embedding(cat_num_embeddings[c], embedding_dim) for c in cat_cols
        })
        # GRU and classifier head
        self.gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, input_ids):
        # input_ids: tuple (num_tensor, cat_tensor)
        num_tensor, cat_tensor = input_ids
        B = num_tensor.size(0)
        emb_list = []
        # numeric projection -> produce (B, n_num, embedding_dim)
        if self.n_num > 0:
            num_flat = num_tensor.view(-1,1)                    # (B*n_num,1)
            num_proj_flat = self.num_proj(num_flat)             # (B*n_num, emb_dim)
            num_proj = num_proj_flat.view(B, self.n_num, -1)    # (B, n_num, emb_dim)
            emb_list.append(num_proj)
        # categorical embeddings -> (B, n_cat, emb_dim)
        if self.n_cat > 0:
            cat_embs = []
            for i, c in enumerate(self.cat_cols):
                emb_i = self.cat_embeddings[c](cat_tensor[:, i])   # (B, emb_dim)
                cat_embs.append(emb_i.unsqueeze(1))               # (B,1,emb_dim)
            cat_embs = torch.cat(cat_embs, dim=1)
            emb_list.append(cat_embs)
        # concat into sequence (B, seq_len, emb_dim)
        seq = torch.cat(emb_list, dim=1)
        # GRU forward -> use last hidden state
        _, hidden = self.gru(seq)
        out = self.fc(self.dropout(hidden[-1]))
        return out

## Author cell 29

In [8]:


import time
from tqdm.auto import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

# Select device: use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Optional: if using GPU, set random seed for reproducibility (optional)
import random, numpy as np
random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1234)
    # Display GPU name (for confirmation)
    try:
        print("GPU name:", torch.cuda.get_device_name(0))
    except:
        pass

def train_and_evaluate(hyperparams, train_loader, val_loader, vocab_size, output_dim, device):
    embedding_dim = int(hyperparams['embedding_dim'])
    hidden_dim = int(hyperparams['hidden_dim'])
    lr = float(hyperparams['lr'])
    epochs = int(hyperparams.get('epochs', 5))

    # Initialize the model, loss, and optimizer
    model = GRUSentimentModel(
        num_cols=num_cols,
        cat_cols=cat_cols,
        cat_num_embeddings=cat_num_embeddings,
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim
    ).to(device)

    # **MODIFIED**: Compute class weights for the training set
    all_train_labels = []
    for batch in train_loader:
        lbls = batch['label']
        if isinstance(lbls, torch.Tensor):
            lbls = lbls.cpu().numpy()
        else:
            lbls = np.array(lbls)
        all_train_labels.extend(lbls.tolist())

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(all_train_labels),
        y=all_train_labels
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)  # Convert to PyTorch tensor

    # **MODIFIED**: Use CrossEntropyLoss with class weights
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training loop
    for epoch in range(hyperparams['epochs']):
        model.train()
        total_loss = 0
        train_loop = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{epochs}", leave=True, position=0)
        for batch in train_loop:
            num_tensor, cat_tensor = batch['input_ids']
            num_tensor = num_tensor.to(device).float()
            cat_tensor = cat_tensor.to(device).long()
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model((num_tensor, cat_tensor))
            loss = criterion(outputs, labels)  # **MODIFIED**: Use CrossEntropyLoss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

            train_loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

    # Validation evaluation
    model.eval()
    val_predictions, val_labels = [], []
    val_loop = tqdm(val_loader, desc="Validating", leave=True, position=0)
    with torch.no_grad():
        for batch in val_loop:
            num_tensor, cat_tensor = batch['input_ids']
            num_tensor = num_tensor.to(device).float()
            cat_tensor = cat_tensor.to(device).long()
            labels = batch['label'].to(device)

            outputs = model((num_tensor, cat_tensor))
            preds = torch.argmax(outputs, dim=1)
            val_predictions.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    # Calculate F1 score and accuracy
    f1 = f1_score(val_labels, val_predictions, average='weighted')
    accuracy = accuracy_score(val_labels, val_predictions)
    print(f"Validation F-1 Score: {f1:.4f}, Accuracy: {accuracy:.4f}")
    return f1, accuracy, model


def random_hyperparameter_tuning(train_loader, val_loader, vocab_size, output_dim, device, n_iter=20):
    # Define hyperparameter ranges for random search
    param_space = {
        'embedding_dim': (150, 250),
        'hidden_dim': (256, 768),
        'lr': (np.log10(1e-4), np.log10(1e-3)),  # Log-uniform for learning rate
        'epochs': (5, 10),  # **MODIFIED**: Removed `alpha` and `gamma` since FocalLoss is no longer used
    }

    best_f1 = 0
    best_params = None
    best_model = None

    # Random search iterations
    with tqdm(total=n_iter, desc="Random Search Tuning", leave=True, position=0) as pbar:
        for _ in range(n_iter):
            # Sample hyperparameters
            hyperparams = {
                'embedding_dim': np.random.uniform(*param_space['embedding_dim']),
                'hidden_dim': np.random.uniform(*param_space['hidden_dim']),
                'lr': 10**np.random.uniform(*param_space['lr']),
                'epochs': np.random.randint(*param_space['epochs']),
            }
            f1, accuracy, model = train_and_evaluate(hyperparams, train_loader, val_loader, vocab_size, output_dim, device)
            pbar.set_postfix(f1=f1, accuracy=accuracy)
            pbar.update(1)

            if f1 > best_f1:
                best_f1 = f1
                best_params = hyperparams
                best_model = model

    print(f"Best Validation F1 Score: {best_f1:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    # Save the best model
    torch.save(best_model.state_dict(), "best_gru_model.pth")
    return best_params, best_f1, best_model

_source_train_and_evaluate = train_and_evaluate
replay_trials = []


def _observed_train_and_evaluate(*args, **kwargs):
    trial_started = time.perf_counter()
    result = _source_train_and_evaluate(*args, **kwargs)
    params = args[0] if args else kwargs["hyperparams"]
    row = {
        "trial": len(replay_trials) + 1,
        "hyperparams": {
            key: value.item() if isinstance(value, np.generic) else value
            for key, value in params.items()
        },
        "validation_f1_weighted": float(result[0]),
        "validation_accuracy": float(result[1]),
        "seconds": time.perf_counter() - trial_started,
    }
    replay_trials.append(row)
    Path("hpo_trials.json").write_text(
        json.dumps(replay_trials, indent=2), encoding="utf-8"
    )
    print("HPO_TRIAL", json.dumps(row), flush=True)
    return result


train_and_evaluate = _observed_train_and_evaluate
# Start timer
start_time = time.time()
# Run random search
best_params, best_f1, best_model= random_hyperparameter_tuning(
    train_loader_3,
    val_loader_3,
    vocab_size=None,
    output_dim=2,
    device=device,
    n_iter=10  # Number of random samples to evaluate
)
# End timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total Parameter Tuning Time: {total_time:.2f} seconds")


Using device: cuda
GPU name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Epoch 1/5, Loss: 0.4925
Epoch 2/5, Loss: 0.4706
Epoch 3/5, Loss: 0.4606
Epoch 4/5, Loss: 0.4460
Epoch 5/5, Loss: 0.4318
Validation F-1 Score: 0.8772, Accuracy: 0.7973
HPO_TRIAL {"trial": 1, "hyperparams": {"embedding_dim": 169.15194503788922, "hidden_dim": 574.5196907723939, "lr": 0.0002739856006447883, "epochs": 5}, "validation_f1_weighted": 0.8771894401582767, "validation_accuracy": 0.797275, "seconds": 334.4066132000007}
Epoch 1/8, Loss: 0.4924
Epoch 2/8, Loss: 0.4702
Epoch 3/8, Loss: 0.4607
Epoch 4/8, Loss: 0.4513
Epoch 5/8, Loss: 0.4394
Epoch 6/8, Loss: 0.4254
Epoch 7/8, Loss: 0.4200
Epoch 8/8, Loss: 0.4131
Validation F-1 Score: 0.9012, Accuracy: 0.8367
HPO_TRIAL {"trial": 2, "hyperparams": {"embedding_dim": 227.1359918862401, "hidden_dim": 696.6629229453758, "lr": 0.00014146107875525552, "epochs": 8}, "validation_f1_weighted": 0.9012319895732324, "validation_accuracy": 0.83665625, "seconds": 602.4424449999988}
Ep

## Author cell 30

In [9]:


# ===========Fixed hyperparameters (random search results)==============
hyperparams = {
    'embedding_dim': 186.88240060019746,
    'hidden_dim': 733.7677322150511,
    'lr': 0.000448103301026688,
    'epochs': 8
}

train_loader = train_loader_3
val_loader=val_loader_3

output_dim=2
device=device
embedding_dim = int(hyperparams['embedding_dim'])
hidden_dim = int(hyperparams['hidden_dim'])
lr = float(hyperparams['lr'])


# Initialize the model, loss, and optimizer
model = GRUSentimentModel(
        num_cols=num_cols,
        cat_cols=cat_cols,
        cat_num_embeddings=cat_num_embeddings,
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim
    ).to(device)

# =====Class weights (addressing severe imbalance)=====
all_train_labels = []
for batch in train_loader:
    all_train_labels.extend(batch['label'].numpy())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_train_labels),
    y=all_train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)  # Convert to PyTorch tensor

# =====Define loss function and optimizer=====
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# =====Training + validation loop=====
for epoch in range(hyperparams['epochs']):
    model.train()
    total_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{hyperparams['epochs']}", leave=True, position=0)
    for batch in train_loop:
        num_tensor, cat_tensor = batch['input_ids']
        num_tensor = num_tensor.to(device).float()
        cat_tensor = cat_tensor.to(device).long()
        labels = batch['label'].to(device).long()

        optimizer.zero_grad()
        outputs = model((num_tensor, cat_tensor))
        loss = criterion(outputs, labels)  # **MODIFIED**: Use CrossEntropyLoss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        train_loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch + 1}/{hyperparams['epochs']}, Loss: {total_loss / len(train_loader):.4f}")

Epoch 1/8, Loss: 0.4925
Epoch 2/8, Loss: 0.4681
Epoch 3/8, Loss: 0.4408
Epoch 4/8, Loss: 0.4328
Epoch 5/8, Loss: 0.4289
Epoch 6/8, Loss: 0.4231
Epoch 7/8, Loss: 0.4183
Epoch 8/8, Loss: 0.4096


## Author cell 31

In [10]:
model_save_path = "GRU_Binary.pth"
torch.save(model.state_dict(), model_save_path)

## Author cell 33

In [11]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
# Evaluate the best model on the test set
best_model = model
best_model.eval()
test_predictions, test_labels = [], []

# **MODIFIED**: Added probabilities collection for further analysis if needed
test_probabilities = []
test_loop = tqdm(test_loader_3, desc="Testing Best Model", leave=True, position=0)
with torch.no_grad():
    for batch in test_loop:
        num_tensor, cat_tensor = batch['input_ids']
        num_tensor = num_tensor.to(device).float()
        cat_tensor = cat_tensor.to(device).long()
        labels = batch['label'].to(device).long()

        # Forward pass
        outputs = best_model((num_tensor, cat_tensor))

        # **MODIFIED**: Added probability calculation using softmax
        probs = torch.softmax(outputs, dim=1)  # Probabilities for each class
        preds = torch.argmax(probs, dim=1)  # Predicted class

        test_predictions.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

        # **MODIFIED**: Collect probabilities of the positive class (e.g., class 1)
        test_probabilities.extend(probs[:, 1].cpu().numpy())

# Calculate F1 score and accuracy on the test set
test_f1 = f1_score(test_labels, test_predictions, average='weighted')
test_accuracy = accuracy_score(test_labels, test_predictions)

print(f"Test F-1 Score: {test_f1:.4f}, Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_predictions, digits=4))


Test F-1 Score: 0.8503, Test Accuracy: 0.7551

Classification Report:
              precision    recall  f1-score   support

           0     0.9977    0.7541    0.8590    197794
           1     0.0369    0.8454    0.0708      2206

    accuracy                         0.7551    200000
   macro avg     0.5173    0.7998    0.4649    200000
weighted avg     0.9871    0.7551    0.8503    200000



## Save evidence after original test cell

In [12]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

scores = np.asarray(test_probabilities, dtype=np.float32)
labels = np.asarray(test_labels, dtype=np.int64)
predictions = np.asarray(test_predictions, dtype=np.int64)
if not np.array_equal(labels, y_test.to_numpy(dtype=np.int64)):
    raise ValueError("Source test labels do not match split row order")
tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
metrics = {
    "fraud_precision": float(precision_score(labels, predictions, zero_division=0)),
    "fraud_recall": float(recall_score(labels, predictions)),
    "fraud_f1": float(f1_score(labels, predictions)),
    "weighted_f1": float(f1_score(labels, predictions, average="weighted")),
    "weighted_recall": float(recall_score(labels, predictions, average="weighted")),
    "accuracy": float(accuracy_score(labels, predictions)),
    "roc_auc": float(roc_auc_score(labels, scores)),
    "ap": float(average_precision_score(labels, scores)),
    "fpr": float(fp / (fp + tn)),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
}
if not np.isclose(metrics["weighted_f1"], test_f1) or not np.isclose(
    metrics["accuracy"], test_accuracy
):
    raise ValueError("Source metrics disagree with saved predictions")
pd.DataFrame(
    {
        "row_index": y_test.index,
        "y_true": labels,
        "score": scores,
        "prediction_argmax": predictions,
    }
).to_csv("test_predictions.csv.gz", index=False)
Path("test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
Path("classification_report.json").write_text(
    json.dumps(
        classification_report(labels, predictions, output_dict=True, zero_division=0),
        indent=2,
    ),
    encoding="utf-8",
)

splits = [
    ("train", X_train_df, y_train_df),
    ("validation", X_val_df, y_val_df),
    ("random_test", X_test, y_test),
]
split_table = pd.concat(
    [
        pd.DataFrame(
            {
                "partition": name,
                "row_index": x.index,
                "y_true": y.to_numpy(dtype=np.int64),
            }
        )
        for name, x, y in splits
    ]
)
if split_table.row_index.duplicated().any() or len(split_table) != EXPECTED_ROWS:
    raise ValueError("Overlapping or incomplete original splits")
split_table.to_csv("split_indices.csv.gz", index=False)
preprocessing = {
    "numeric_columns": num_cols,
    "categorical_columns": cat_cols,
    "fit_rows": len(X_train),
    "imputer_statistics": num_pipeline.named_steps["imputer"].statistics_.tolist(),
    "scaler_mean": num_pipeline.named_steps["scaler"].mean_.tolist(),
    "scaler_scale": num_pipeline.named_steps["scaler"].scale_.tolist(),
    "cat_to_index": cat_to_index,
}
Path("preprocessing.json").write_text(
    json.dumps(preprocessing, indent=2), encoding="utf-8"
)
Path("source_results.json").write_text(
    json.dumps(
        {
            "hpo_trials": len(replay_trials),
            "best_params": best_params,
            "best_validation_f1_weighted": float(best_f1),
            "fixed_hyperparams": hyperparams,
            "class_weights": class_weights.tolist(),
            "source_test_f1_weighted": float(test_f1),
            "source_test_accuracy": float(test_accuracy),
            "split_rows": {name: len(x) for name, x, y in splits},
        },
        default=lambda value: (
            value.item() if isinstance(value, np.generic) else str(value)
        ),
        indent=2,
    ),
    encoding="utf-8",
)
print("ORIGINAL_GRU_ARTIFACTS_READY")
print(json.dumps(metrics, indent=2))


ORIGINAL_GRU_ARTIFACTS_READY
{
  "fraud_precision": 0.0369350813958094,
  "fraud_recall": 0.8454215775158658,
  "fraud_f1": 0.07077798861480077,
  "weighted_f1": 0.8503039178974042,
  "weighted_recall": 0.75515,
  "accuracy": 0.75515,
  "roc_auc": 0.8822224491994386,
  "ap": 0.1573626938623287,
  "fpr": 0.2458568005096211,
  "tn": 149165,
  "fp": 48629,
  "fn": 341,
  "tp": 1865
}
